In [ ]:
from google.colab import drive
import os
import shutil

# Ensure the mount point is clean before attempting to mount
# Check if the directory exists and remove it if it's a non-empty directory
if os.path.exists("/content/drive") and os.path.isdir("/content/drive"):
    print("Removing existing /content/drive directory...")
    shutil.rmtree("/content/drive")

# Recreate the directory to serve as the mount point
os.makedirs("/content/drive", exist_ok=True)

drive.mount("/content/drive", force_remount=True)

In [ ]:
import os
import json
import shutil
import zipfile
import hashlib
import random
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from PIL import Image, ImageOps
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

In [ ]:
ZIP_PATH = Path(
    "/content/drive/MyDrive/CinnamonAI/dataset/research_images.zip"
)

LOCAL_ZIP_PATH = Path("/content/research_images.zip")
EXTRACT_PATH = Path("/content/cinnamon_raw")
CLEAN_PATH = Path("/content/cinnamon_clean")
SPLIT_PATH = Path("/content/cinnamon_split")

MODEL_DRIVE_PATH = Path(
    "/content/drive/MyDrive/CinnamonAI/trained_models"
)

MODEL_DRIVE_PATH.mkdir(parents=True, exist_ok=True)

if not ZIP_PATH.exists():
    raise FileNotFoundError(
        f"ZIP file not found at:\n{ZIP_PATH}\n"
        "Check the Google Drive folder and filename."
    )

print("Dataset ZIP found:", ZIP_PATH)

In [ ]:
if LOCAL_ZIP_PATH.exists():
    LOCAL_ZIP_PATH.unlink()

shutil.copy2(ZIP_PATH, LOCAL_ZIP_PATH)

print(
    "ZIP copied to Colab:",
    LOCAL_ZIP_PATH,
    f"({LOCAL_ZIP_PATH.stat().st_size / 1024**2:.2f} MB)"
)

In [ ]:
if EXTRACT_PATH.exists():
    shutil.rmtree(EXTRACT_PATH)

EXTRACT_PATH.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(LOCAL_ZIP_PATH, "r") as zip_file:
    zip_file.extractall(EXTRACT_PATH)

print("Dataset extracted successfully.")

In [ ]:
matching_directories = list(
    EXTRACT_PATH.rglob("leaves_diseases")
)

if not matching_directories:
    raise FileNotFoundError(
        "Could not find the leaves_diseases folder "
        "inside the extracted ZIP file."
    )

DATASET_ROOT = matching_directories[0]

print("Dataset root:", DATASET_ROOT)

In [ ]:
class_directories = sorted(
    directory
    for directory in DATASET_ROOT.iterdir()
    if directory.is_dir()
)

class_names_from_folders = [
    directory.name
    for directory in class_directories
]

print("Classes detected:")

for index, class_name in enumerate(class_names_from_folders):
    print(f"{index}: {class_name}")

print("\nNumber of classes:", len(class_names_from_folders))

In [ ]:
SUPPORTED_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".gif",
    ".webp",
    ".tif",
    ".tiff",
}

original_class_counts = {}

for class_directory in class_directories:
    image_paths = [
        path
        for path in class_directory.rglob("*")
        if path.is_file()
        and path.suffix.lower() in SUPPORTED_EXTENSIONS
    ]

    original_class_counts[class_directory.name] = len(image_paths)

print("Images per class:\n")

for class_name, count in original_class_counts.items():
    print(f"{class_name:25s}: {count}")

print("\nTotal images:", sum(original_class_counts.values()))

In [ ]:
if CLEAN_PATH.exists():
    shutil.rmtree(CLEAN_PATH)

CLEAN_PATH.mkdir(parents=True, exist_ok=True)

converted_count = 0
corrupted_files = []
duplicate_files = []
label_conflicts = []

# Stores normalized image hashes
seen_hashes = {}

for class_directory in class_directories:
    destination_class = CLEAN_PATH / class_directory.name
    destination_class.mkdir(parents=True, exist_ok=True)

    image_paths = [
        path
        for path in class_directory.rglob("*")
        if path.is_file()
        and path.suffix.lower() in SUPPORTED_EXTENSIONS
    ]

    for image_number, image_path in enumerate(image_paths):
        try:
            with Image.open(image_path) as image:
                # Correct phone-camera EXIF rotation
                image = ImageOps.exif_transpose(image)

                # Convert grayscale/RGBA/palette images to RGB
                image = image.convert("RGB")

                # Create a normalized pixel hash
                image_array = np.asarray(image)
                pixel_hash = hashlib.sha256(
                    image_array.tobytes()
                    + str(image.size).encode()
                ).hexdigest()

                if pixel_hash in seen_hashes:
                    previous_class, previous_path = seen_hashes[pixel_hash]

                    if previous_class != class_directory.name:
                        label_conflicts.append({
                            "image": str(image_path),
                            "current_class": class_directory.name,
                            "previous_image": previous_path,
                            "previous_class": previous_class,
                        })
                    else:
                        duplicate_files.append(str(image_path))

                    continue

                seen_hashes[pixel_hash] = (
                    class_directory.name,
                    str(image_path),
                )

                destination_path = (
                    destination_class
                    / f"{class_directory.name}_{converted_count:07d}.jpg"
                )

                image.save(
                    destination_path,
                    format="JPEG",
                    quality=95,
                    optimize=True,
                )

                converted_count += 1

        except Exception as error:
            corrupted_files.append({
                "path": str(image_path),
                "error": str(error),
            })

print("Clean images created:", converted_count)
print("Same-class duplicates skipped:", len(duplicate_files))
print("Corrupted/unreadable images:", len(corrupted_files))
print("Cross-class label conflicts:", len(label_conflicts))

In [ ]:
for conflict in label_conflicts[:20]:
    print("\nPossible label conflict:")
    print("Current:", conflict["image"])
    print("Current class:", conflict["current_class"])
    print("Previous:", conflict["previous_image"])
    print("Previous class:", conflict["previous_class"])

In [ ]:
for item in corrupted_files[:20]:
    print(item["path"], "->", item["error"])

In [ ]:
clean_class_directories = sorted(
    directory
    for directory in CLEAN_PATH.iterdir()
    if directory.is_dir()
)

clean_class_counts = {}

for class_directory in clean_class_directories:
    count = len(list(class_directory.glob("*.jpg")))
    clean_class_counts[class_directory.name] = count

print("Clean dataset counts:\n")

for class_name, count in clean_class_counts.items():
    print(f"{class_name:25s}: {count}")

print("\nTotal clean images:", sum(clean_class_counts.values()))

Phase 7 — Create train, validation and test sets

70% training
15% validation
15% internal testing

In [ ]:
from sklearn.model_selection import train_test_split
import shutil

if SPLIT_PATH.exists():
    shutil.rmtree(SPLIT_PATH)

for split_name in ["train", "validation", "test"]:
    (SPLIT_PATH / split_name).mkdir(
        parents=True,
        exist_ok=True,
    )

MINIMUM_IMAGES = 10
split_summary = {}

for class_directory in clean_class_directories:
    class_name = class_directory.name
    image_paths = sorted(class_directory.glob("*.jpg"))

    if len(image_paths) < MINIMUM_IMAGES:
        raise ValueError(
            f"Class '{class_name}' has only "
            f"{len(image_paths)} usable images. "
            f"At least {MINIMUM_IMAGES} are required."
        )

    train_paths, temporary_paths = train_test_split(
        image_paths,
        test_size=0.30,
        random_state=SEED,
        shuffle=True,
    )

    validation_paths, test_paths = train_test_split(
        temporary_paths,
        test_size=0.50,
        random_state=SEED,
        shuffle=True,
    )

    split_groups = {
        "train": train_paths,
        "validation": validation_paths,
        "test": test_paths,
    }

    split_summary[class_name] = {}

    for split_name, paths in split_groups.items():
        destination_directory = (
            SPLIT_PATH / split_name / class_name
        )

        destination_directory.mkdir(
            parents=True,
            exist_ok=True,
        )

        for source_path in paths:
            shutil.copy2(
                source_path,
                destination_directory / source_path.name,
            )

        split_summary[class_name][split_name] = len(paths)

for class_name, counts in split_summary.items():
    print(
        f"{class_name:25s} "
        f"Train={counts['train']:4d} | "
        f"Validation={counts['validation']:4d} | "
        f"Test={counts['test']:4d}"
    )

Phase 8 — Load the TensorFlow datasets
 Create training, validation and test datasets

In [ ]:
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

train_dataset = tf.keras.utils.image_dataset_from_directory(
    SPLIT_PATH / "train",
    labels="inferred",
    label_mode="int",
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED,
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    SPLIT_PATH / "validation",
    labels="inferred",
    label_mode="int",
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

test_dataset = tf.keras.utils.image_dataset_from_directory(
    SPLIT_PATH / "test",
    labels="inferred",
    label_mode="int",
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

class_names = train_dataset.class_names
NUMBER_OF_CLASSES = len(class_names)

print("\nTensorFlow class order:")

for index, class_name in enumerate(class_names):
    print(index, class_name)

print("\nNumber of classes:", NUMBER_OF_CLASSES)

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.prefetch(
    buffer_size=AUTOTUNE
)

validation_dataset = validation_dataset.prefetch(
    buffer_size=AUTOTUNE
)

test_dataset = test_dataset.prefetch(
    buffer_size=AUTOTUNE
)

In [ ]:
plt.figure(figsize=(12, 10))

for images, labels in train_dataset.take(1):
    for index in range(min(12, len(images))):
        plt.subplot(3, 4, index + 1)
        plt.imshow(
            images[index].numpy().astype("uint8")
        )
        plt.title(
            class_names[int(labels[index])]
        )
        plt.axis("off")

plt.tight_layout()
plt.show()

Phase 9 — Handle class imbalance
Calculate class weights

In [ ]:
training_counts = []

for class_name in class_names:
    class_directory = (
        SPLIT_PATH / "train" / class_name
    )

    count = len(list(class_directory.glob("*.jpg")))
    training_counts.append(count)

training_counts = np.array(training_counts)
total_training_images = training_counts.sum()

class_weights = {
    class_index: (
        total_training_images
        / (
            NUMBER_OF_CLASSES
            * class_count
        )
    )
    for class_index, class_count
    in enumerate(training_counts)
}

print("Training image counts:")

for class_index, class_name in enumerate(class_names):
    print(
        class_index,
        class_name,
        training_counts[class_index],
        "weight:",
        round(class_weights[class_index], 4),
    )

Phase 10 — Create the model

Configure data augmentation

In [ ]:
data_augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip(
            mode="horizontal"
        ),
        tf.keras.layers.RandomRotation(
            factor=0.08
        ),
        tf.keras.layers.RandomZoom(
            height_factor=0.12,
            width_factor=0.12,
        ),
        tf.keras.layers.RandomTranslation(
            height_factor=0.08,
            width_factor=0.08,
        ),
        tf.keras.layers.RandomContrast(
            factor=0.15
        ),
    ],
    name="data_augmentation",
)

In [ ]:
base_model = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=IMAGE_SIZE + (3,),
)

base_model.trainable = False

inputs = tf.keras.Input(
    shape=IMAGE_SIZE + (3,),
    name="input_image",
)

x = data_augmentation(
    inputs,
    training=True,
)

x = base_model(
    x,
    training=False,
)

x = tf.keras.layers.GlobalAveragePooling2D(
    name="global_average_pooling"
)(x)

x = tf.keras.layers.BatchNormalization(
    name="classification_batch_norm"
)(x)

x = tf.keras.layers.Dropout(
    rate=0.35,
    name="classification_dropout"
)(x)

outputs = tf.keras.layers.Dense(
    NUMBER_OF_CLASSES,
    activation="softmax",
    name="disease_predictions",
)(x)

model = tf.keras.Model(
    inputs=inputs,
    outputs=outputs,
    name="cinnamon_leaf_condition_model",
)

model.summary()

NameError: name 'tf' is not defined

In [ ]:
NUMBER_OF_CLASSES = len(class_names)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-3
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        ),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=min(3, NUMBER_OF_CLASSES),
            name="top_3_accuracy",
        ),
    ],
)

NameError: name 'model' is not defined

Phase 11 — Train the first stage

In [ ]:
BEST_MODEL_PATH = (
    MODEL_DRIVE_PATH
    / "best_cinnamon_leaf_model.keras"
)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(BEST_MODEL_PATH),
        monitor="val_loss",
        save_best_only=True,
        verbose=1,
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=6,
        restore_best_weights=True,
        verbose=1,
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=3,
        min_lr=1e-7,
        verbose=1,
    ),
]

Train the classifier head

In [ ]:
INITIAL_EPOCHS = 20

initial_history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=INITIAL_EPOCHS,
    class_weight=class_weights,
    callbacks=callbacks,
)

In [ ]:
import tensorflow as tf
from pathlib import Path

# Define MODEL_DRIVE_PATH and BEST_MODEL_PATH to ensure they are available
MODEL_DRIVE_PATH = Path(
    "/content/drive/MyDrive/CinnamonAI/trained_models"
)
BEST_MODEL_PATH = (
    MODEL_DRIVE_PATH
    / "best_cinnamon_leaf_model.keras"
)

KERAS_MODEL_PATH = BEST_MODEL_PATH

original_model = tf.keras.models.load_model(
    KERAS_MODEL_PATH,
    compile=False
)

print("Original model loaded successfully.")
original_model.summary()

In [ ]:
for index, layer in enumerate(original_model.layers):
    print(
        index,
        layer.name,
        type(layer).__name__
    )

In [ ]:
IMAGE_SIZE = (224, 224)

base_model = original_model.get_layer(
    "efficientnetb0"
)

global_pool = original_model.get_layer(
    "global_average_pooling"
)

classification_bn = original_model.get_layer(
    "classification_batch_norm"
)

classification_dropout = original_model.get_layer(
    "classification_dropout"
)

disease_predictions = original_model.get_layer(
    "disease_predictions"
)

In [ ]:
inputs = tf.keras.Input(
    shape=IMAGE_SIZE + (3,),
    name="input_image"
)

x = base_model(
    inputs,
    training=False
)

x = global_pool(x)

x = classification_bn(
    x,
    training=False
)

x = classification_dropout(
    x,
    training=False
)

outputs = disease_predictions(x)

inference_model = tf.keras.Model(
    inputs=inputs,
    outputs=outputs,
    name="cinnamon_disease_inference_model"
)

inference_model.summary()

In [ ]:
import numpy as np

dummy_image = np.random.randint(
    0,
    256,
    size=(1, 224, 224, 3)
).astype("float32")

prediction = inference_model.predict(
    dummy_image,
    verbose=0
)

print("Prediction shape:", prediction.shape)
print("Prediction:", prediction)
print(
    "Probability sum:",
    prediction[0].sum()
)

In [ ]:
H5_MODEL_PATH = (
    MODEL_FOLDER
    / "cinnamon_disease_model.h5"
)

inference_model.save(
    H5_MODEL_PATH,
    include_optimizer=False
)

print("H5 model created.")
print(H5_MODEL_PATH)

In [ ]:
print(
    "H5 exists:",
    H5_MODEL_PATH.exists()
)

if H5_MODEL_PATH.exists():
    print(
        "H5 size:",
        round(
            H5_MODEL_PATH.stat().st_size
            / 1024**2,
            2
        ),
        "MB"
    )

In [ ]:
h5_model = tf.keras.models.load_model(
    H5_MODEL_PATH,
    compile=False
)

print("H5 model loaded successfully.")

h5_model.summary()

In [ ]:
prediction_before_save = inference_model.predict(
    dummy_image,
    verbose=0
)

prediction_after_save = h5_model.predict(
    dummy_image,
    verbose=0
)

maximum_difference = np.max(
    np.abs(
        prediction_before_save
        - prediction_after_save
    )
)

print(
    "Maximum difference:",
    maximum_difference
)

In [ ]:
class_names = [
    "healthy_cinnamon",
    "leaf_blight",
    "leaf_miner_attack",
    "leaf_patches_fungal",
    "lower_leaf_gall",
    "non_cinnamon",
    "upper_leaf_gall",
]

print(class_names)
print("Number of classes:", len(class_names))

In [ ]:
from pathlib import Path

TEST_DIR = Path("/content/cinnamon_split/test")

if TEST_DIR.exists():
    class_names = sorted([
        folder.name
        for folder in TEST_DIR.iterdir()
        if folder.is_dir()
    ])

    print("Classes recovered from dataset:")
    for i, name in enumerate(class_names):
        print(i, name)
else:
    print("Test dataset directory not found.")

In [ ]:
print("H5 output shape:", h5_model.output_shape)
print("Number of classes:", len(class_names))

assert h5_model.output_shape[-1] == len(class_names)

print("✅ Model output and class count match.")

NameError: name 'h5_model' is not defined

In [ ]:
import json
from pathlib import Path

MODEL_FOLDER = Path(
    "/content/drive/MyDrive/CinnamonAI/trained_models"
)

CLASS_NAMES_PATH = (
    MODEL_FOLDER / "class_names.json"
)

class_names = [
    "healthy_cinnamon",
    "leaf_blight",
    "leaf_miner_attack",
    "leaf_patches_fungal",
    "lower_leaf_gall",
    "non_cinnamon",
    "upper_leaf_gall"
]

with open(
    CLASS_NAMES_PATH,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        class_names,
        file,
        indent=2
    )

print("✅ class_names.json saved")
print(CLASS_NAMES_PATH)

load and test


In [ ]:
import tensorflow as tf
import json
from pathlib import Path

MODEL_FOLDER = Path(
    "/content/drive/MyDrive/CinnamonAI/trained_models"
)

H5_MODEL_PATH = (
    MODEL_FOLDER / "cinnamon_disease_model.h5"
)

CLASS_NAMES_PATH = (
    MODEL_FOLDER / "class_names.json"
)

model = tf.keras.models.load_model(
    H5_MODEL_PATH,
    compile=False
)

with open(
    CLASS_NAMES_PATH,
    "r",
    encoding="utf-8"
) as file:
    class_names = json.load(file)

print("Model loaded successfully.")
print("Output shape:", model.output_shape)
print("Classes:", class_names)

In [ ]:
from google.colab import files

uploaded = files.upload()

image_path = next(iter(uploaded.keys()))

print("Uploaded image:", image_path)

TypeError: 'NoneType' object is not subscriptable

In [ ]:
import numpy as np

IMAGE_SIZE = (224, 224)

image = tf.keras.utils.load_img(
    image_path,
    target_size=IMAGE_SIZE,
    color_mode="rgb"
)

image_array = tf.keras.utils.img_to_array(
    image
)

image_batch = np.expand_dims(
    image_array,
    axis=0
)

probabilities = model.predict(
    image_batch,
    verbose=0
)[0]

sorted_indices = np.argsort(
    probabilities
)[::-1]

In [ ]:
print("\nPrediction probabilities:\n")

for index in sorted_indices:
    print(
        f"{class_names[index]:25s}"
        f" {probabilities[index] * 100:.2f}%"
    )

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

best_index = int(
    np.argmax(probabilities)
)

predicted_class = class_names[
    best_index
]

confidence = float(
    probabilities[best_index]
)

plt.figure(figsize=(7, 7))

plt.imshow(
    Image.open(image_path)
)

plt.axis("off")

plt.title(
    f"Prediction: {predicted_class}\n"
    f"Confidence: {confidence * 100:.2f}%"
)

plt.show()